# 用 Reflect 檢查回答是否有依據

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 載入流程元件

這一章不是再教一次 keyword 設定。`KeywordRetrieve` 只是一個最小 evidence source；真正要看的，是 `Reflect` 如何在 action 回答後留下可檢查的驗收結果。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, EvidenceCheckReflect, KeywordRetrieve, PassThroughPerceive

## 用最小資料建立 evidence source

資料設定刻意保持很小，因為本章重點不是搜尋技巧，而是讓 retrieve 階段明確回報有沒有命中 evidence。

In [ ]:
reference_items = [
    {'keywords': ['保存', 'bundle', '參考文件'], 'content': '使用參考文件的 Agent 需要保存 bundle，重新打開時才找得到原本的資料。'},
    {'keywords': ['runner', '唯讀'], 'content': '公開分享的 Runner 可以使用 Agent，但不能修改設定或保存。'},
]

retrieve = KeywordRetrieve(items=reference_items, fallback='目前沒有找到相關參考資料。')

## 加上 Reflect 驗收關卡

`EvidenceCheckReflect` 會在 action 之後檢查流程狀態。這個範例把 `on_failure` 設成 `end`，所以沒有 evidence 時不會進入重試，而是把 `reflect_verdict` 留在結果裡讓你檢查。

In [ ]:
workflow = Workflow(
    workflow_name='Reflect evidence check Agent',
    perceive=PassThroughPerceive(),
    retrieve=retrieve,
    action=DirectAnswerAction(),
    reflect=EvidenceCheckReflect(on_failure='end'),
)

## Case 1：有 evidence，Reflect 通過

這個問題會命中 bundle 條目。回答完成後，不只看 `final_message`，也要看 `reflect_verdict` 是否為 `pass`。

In [ ]:
grounded_result = workflow.run('為什麼使用參考文件的 Agent 要保存 bundle？')
print(grounded_result.final_message)
print('reflect verdict:', grounded_result.entities.get('reflect_verdict'))

## 看 Reflect 留下的驗收紀錄

`reflect_verdict` 適合給程式判斷；reflection entry 則適合除錯，因為它會記錄 verdict 和 reason。

In [ ]:
reflection_entries = [entry for entry in grounded_result.entries if entry.type == 'reflection']
print(reflection_entries[-1].content)
print(reflection_entries[-1].metadata)

## Case 2：沒有 evidence，Reflect 擋下來

這個問題不在參考資料裡。Action 仍會產生 fallback 訊息，但 Reflect 會把 verdict 標成 `fail`，讓應用層可以把它視為未通過 evidence gate 的回答。

In [ ]:
missing_result = workflow.run('這個 Agent 支援哪些 GPU driver 版本？')
print(missing_result.final_message)
print('reflect verdict:', missing_result.entities.get('reflect_verdict'))

## 這一章的重點

01 已經示範如何把 perceive、retrieve、action 串起來。04 要補上的觀念是：retrieve 不只是拿資料給 action，它也提供 evidence 訊號給 Reflect 驗收；正式換成文件 bundle、語意搜尋或向量索引時，這個 evidence gate 仍然是同一個位置。

In [ ]:
for entry in missing_result.entries:
    if entry.type in {'retrieved', 'reflection'}:
        print(entry.type, entry.content, entry.metadata)